# 🦷 Fine-tune MedGemma for Dental Diagnostics

This notebook fine-tunes **[MedGemma 1.5 4B IT](https://huggingface.co/google/medgemma-1.5-4b-it)** on dental X-ray analysis using the **[DentalGemma](https://huggingface.co/datasets/naazimsnh02/dentalgemma-vqa)** datasets — a novel domain adaptation since MedGemma was **not** trained on dental data.

**Fine-tuning approach:**
- [QLoRA](https://arxiv.org/abs/2305.14314) (4-bit quantization + LoRA) for memory-efficient training
- [TRL SFTTrainer](https://github.com/huggingface/trl) for supervised fine-tuning
- Two-stage training: multimodal VQA (image + text) → text-only clinical instruct data

**Datasets:**
- [`naazimsnh02/dentalgemma-vqa`](https://huggingface.co/datasets/naazimsnh02/dentalgemma-vqa) — 1,654 dental X-ray VQA pairs
- [`naazimsnh02/dentalgemma-instruct`](https://huggingface.co/datasets/naazimsnh02/dentalgemma-instruct) — 2,494 clinical dental cases

**Built for the [MedGemma Impact Challenge](https://kaggle.com/competitions/med-gemma-impact-challenge).**

## Setup

**Requirements:** A GPU with bfloat16 support and ≥40 GB VRAM (e.g., A100).

**Google Colab:**
1. Click **▾ (Additional connection options)** → **Change runtime type**
2. Select **A100 GPU** under Hardware accelerator

### Get access to MedGemma

1. Create a [Hugging Face account](https://huggingface.co/join) if you don't have one
2. Accept usage conditions at [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)

### Configure your HF token

Generate a Hugging Face **write** access token at [settings/tokens](https://huggingface.co/settings/tokens).

**Google Colab:** Add your token to the 🔑 Secrets tab as `HF_TOKEN`.

In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

### Install dependencies

In [ ]:
# First, reinstall torch and torchvision to ensure compatible versions
!pip install --upgrade --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu124
!pip install --upgrade --quiet accelerate bitsandbytes datasets evaluate peft tensorboard transformers trl

## Load DentalGemma Datasets

We use two purpose-built datasets from HuggingFace Hub:

| Dataset | Type | Samples | Tasks |
|:--------|:-----|--------:|:------|
| `dentalgemma-vqa` | Multimodal (image + text) | 1,654 | Cavity detection, OPG classification, tooth ID, radiograph assessment |
| `dentalgemma-instruct` | Text-only | 2,494 | Clinical case assessment across 98 dental conditions |

Both datasets are pre-formatted in chat template format (system/user/assistant) compatible with MedGemma's expected input.

In [ ]:
from datasets import load_dataset

vqa_data = load_dataset("naazimsnh02/dentalgemma-vqa")
instruct_data = load_dataset("naazimsnh02/dentalgemma-instruct")

print("=" * 50)
print("DentalGemma VQA (Multimodal):")
print(vqa_data)
print("\nDentalGemma Instruct (Text-only):")
print(instruct_data)

### Inspect VQA samples

Each VQA sample contains:
- `image`: dental X-ray as a PIL Image
- `messages`: JSON string with chat-format messages (system/user[image+text]/assistant)
- `source`: origin dataset tag
- `condition`: dental condition label

In [ ]:
import json
from collections import Counter

sample = vqa_data["train"][0]
print("Image size:", sample["image"].size)
print("Source:", sample["source"])
print("Condition:", sample["condition"])
print("\nMessages:")
messages = json.loads(sample["messages"])
for msg in messages:
    role = msg["role"]
    if isinstance(msg["content"], list):
        types = [c["type"] for c in msg["content"]]
        print(f"  {role}: [{', '.join(types)}]")
    else:
        print(f"  {role}: {msg['content'][:120]}...")

print("\n--- Source distribution (train) ---")
for src, cnt in sorted(Counter(vqa_data["train"]["source"]).items()):
    print(f"  {src}: {cnt}")

In [ ]:
# Display a sample dental X-ray
vqa_data["train"][0]["image"]

### Inspect instruct samples

In [ ]:
sample = instruct_data["train"][0]
print("Source:", sample["source"])
print("Condition:", sample["condition"])
print("\nMessages:")
messages = json.loads(sample["messages"])
for msg in messages:
    print(f"  {msg['role']}: {str(msg['content'])[:150]}...")

print(f"\nUnique conditions: {len(set(instruct_data['train']['condition']))}")

### Format datasets for training

Parse the JSON `messages` field into Python objects for the chat template collator. The VQA dataset has multimodal message format (image + text); the instruct dataset has text-only messages.

**Note:** MedGemma 1.5 expects system content as `[{"type": "text", "text": "..."}]` list format (see the [official quick start notebook](https://github.com/google-health/medgemma)). We normalize our stored string-format system messages to match.

In [ ]:
import json
from typing import Any


def _normalize_messages(messages: list[dict]) -> list[dict]:
    """Ensure system content uses the list-of-dicts format MedGemma expects.

    MedGemma 1.5 chat template expects:
      {"role": "system", "content": [{"type": "text", "text": "..."}]}
    Our datasets store system content as a plain string.
    """
    normalized = []
    for msg in messages:
        if msg["role"] == "system" and isinstance(msg["content"], str):
            normalized.append({
                "role": "system",
                "content": [{"type": "text", "text": msg["content"]}],
            })
        elif msg["role"] == "assistant" and isinstance(msg["content"], str):
            normalized.append({
                "role": "assistant",
                "content": [{"type": "text", "text": msg["content"]}],
            })
        else:
            normalized.append(msg)
    return normalized


def parse_vqa_messages(example: dict[str, Any]) -> dict[str, Any]:
    """Parse JSON messages and normalize for MedGemma's chat template."""
    example["messages"] = _normalize_messages(json.loads(example["messages"]))
    return example


def parse_instruct_messages(example: dict[str, Any]) -> dict[str, Any]:
    """Parse JSON messages and normalize for MedGemma's chat template.

    Instruct messages are text-only, so user content is wrapped in
    [{"type": "text", "text": "..."}] to match the expected format.
    """
    raw = json.loads(example["messages"])
    normalized = []
    for msg in raw:
        if isinstance(msg["content"], str):
            normalized.append({
                "role": msg["role"],
                "content": [{"type": "text", "text": msg["content"]}],
            })
        else:
            normalized.append(msg)
    example["messages"] = normalized
    return example


vqa_data = vqa_data.map(parse_vqa_messages)
instruct_data = instruct_data.map(parse_instruct_messages)

print("VQA train sample messages (parsed):")
sample_msgs = vqa_data['train'][0]['messages']
for m in sample_msgs:
    if isinstance(m['content'], list):
        print(f"  {m['role']}: [{', '.join(c['type'] for c in m['content'])}]")
    else:
        print(f"  {m['role']}: {str(m['content'])[:80]}...")

print(f"\nInstruct train sample messages (parsed):")
sample_msgs = instruct_data['train'][0]['messages']
for m in sample_msgs:
    if isinstance(m['content'], list):
        print(f"  {m['role']}: [{', '.join(c['type'] for c in m['content'])}]")
    else:
        print(f"  {m['role']}: {str(m['content'])[:80]}...")

## Fine-tune with QLoRA

We use **Quantized Low-Rank Adaptation (QLoRA)** to fine-tune MedGemma efficiently:

1. The base model is quantized to **4-bit NF4** precision
2. Small LoRA adapter layers are attached and trained
3. Only the adapter parameters are updated — the base model stays frozen

This allows fine-tuning a 4B parameter model on a single A100 GPU.

### Load MedGemma with 4-bit quantization

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-1.5-4b-it"

if torch.cuda.get_device_capability()[0] < 8:
    raise ValueError("GPU does not support bfloat16. Please use an A100 or newer GPU.")

model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=model_kwargs["torch_dtype"],
    bnb_4bit_quant_storage=model_kwargs["torch_dtype"],
)

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

print(f"Model loaded: {model_id}")
print(f"Quantization: 4-bit NF4 with double quantization")

### Configure LoRA

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

### Data collators

We define separate collators for the multimodal VQA data (with images) and the text-only instruct data. Both use `processor.apply_chat_template` to format messages into the model's expected input format.

In [ ]:
from typing import Any


def vqa_collate_fn(examples: list[dict[str, Any]]):
    """Collator for multimodal VQA examples (image + text)."""
    texts = []
    images = []
    for example in examples:
        images.append([example["image"].convert("RGB")])
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        ).strip())

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch


def instruct_collate_fn(examples: list[dict[str, Any]]):
    """Collator for text-only instruct examples."""
    texts = []
    for example in examples:
        texts.append(processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        ).strip())

    batch = processor.tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=1024
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    batch["labels"] = labels
    return batch

## Stage 1: Fine-tune on VQA Dataset (Multimodal)

First, we train on the multimodal VQA dataset containing dental X-ray images paired with clinical questions and answers. This teaches the model to:
- Detect cavities and count regions in intraoral X-rays
- Classify dental pathologies from panoramic X-rays (6 classes)
- Identify and count tooth types from panoramic radiographs
- Perform systematic radiographic assessments

**Checkpointing:** Model checkpoints are saved every 100 steps and at each epoch to prevent data loss during long training runs.

In [ ]:
from trl import SFTConfig

output_dir = "dentalgemma-1.5-4b-it-vqa"  # @param {type: "string"}
num_train_epochs = 3  # @param {type: "number"}
learning_rate = 2e-4  # @param {type: "number"}

vqa_training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=learning_rate,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="linear",
    push_to_hub=False,
    report_to="tensorboard",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
)

In [ ]:
from trl import SFTTrainer

vqa_trainer = SFTTrainer(
    model=model,
    args=vqa_training_args,
    train_dataset=vqa_data["train"],
    eval_dataset=vqa_data["validation"],
    peft_config=peft_config,
    processing_class=processor,
    data_collator=vqa_collate_fn,
)

### Train on VQA data

Training resumes automatically from the latest checkpoint if one exists in the output directory.

In [ ]:
import os

# Resume from checkpoint if available
vqa_checkpoint = None
if os.path.isdir(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        vqa_checkpoint = os.path.join(output_dir, latest)
        print(f"Resuming VQA training from: {vqa_checkpoint}")

vqa_trainer.train(resume_from_checkpoint=vqa_checkpoint)

In [ ]:
vqa_trainer.save_model()
print(f"VQA adapter saved to: {output_dir}")

## Stage 2: Fine-tune on Instruct Dataset (Text-only)

Next, we continue training on the text-only clinical instruct dataset. This teaches the model structured clinical reasoning:
- Diagnosis from patient history and symptoms
- Treatment planning with management protocols
- Antibiotic considerations
- Follow-up scheduling and patient counseling

The LoRA adapter from Stage 1 is carried forward so the model retains its visual capabilities while gaining clinical text reasoning.

In [ ]:
instruct_output_dir = "dentalgemma-1.5-4b-it-instruct"  # @param {type: "string"}

instruct_training_args = SFTConfig(
    output_dir=instruct_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=1e-4,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="linear",
    push_to_hub=False,
    report_to="tensorboard",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
)

In [ ]:
instruct_trainer = SFTTrainer(
    model=vqa_trainer.model,
    args=instruct_training_args,
    train_dataset=instruct_data["train"],
    eval_dataset=instruct_data["validation"],
    processing_class=processor.tokenizer,
    data_collator=instruct_collate_fn,
)

### Train on instruct data

In [ ]:
# Resume from checkpoint if available
instruct_checkpoint = None
if os.path.isdir(instruct_output_dir):
    checkpoints = [d for d in os.listdir(instruct_output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        instruct_checkpoint = os.path.join(instruct_output_dir, latest)
        print(f"Resuming instruct training from: {instruct_checkpoint}")

instruct_trainer.train(resume_from_checkpoint=instruct_checkpoint)

In [ ]:
instruct_trainer.save_model()
print(f"Instruct adapter saved to: {instruct_output_dir}")

## Merge LoRA Adapter & Push to Hub

Merge the trained LoRA adapter weights into the base model to create a standalone model that can be used without PEFT, then push the merged model and processor to HuggingFace Hub.

In [ ]:
# Free training memory before loading for merge
del vqa_trainer
del instruct_trainer
torch.cuda.empty_cache()
print("Training memory freed.")

In [ ]:
from peft import PeftModel

# Reload the base model in full precision for merging
print("Loading base model for merge...")
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Load the fine-tuned LoRA adapter
print(f"Loading LoRA adapter from: {instruct_output_dir}")
merged_model = PeftModel.from_pretrained(base_model, instruct_output_dir)

# Merge LoRA weights into the base model
print("Merging LoRA adapter into base model...")
merged_model = merged_model.merge_and_unload()
print("Merge complete.")

In [ ]:
hub_repo_id = "naazimsnh02/dentalgemma-1.5-4b-it"  # @param {type: "string"}

print(f"Pushing merged model to: {hub_repo_id}")
merged_model.push_to_hub(hub_repo_id, private=False)
processor.push_to_hub(hub_repo_id)
print(f"\n✅ Model and processor pushed to: https://huggingface.co/{hub_repo_id}")

In [ ]:
del base_model
del merged_model
torch.cuda.empty_cache()

## Evaluate the Fine-Tuned Model

Test the fine-tuned DentalGemma model on the held-out validation set to verify it has learned dental-specific capabilities.

### Load the fine-tuned model

Load the merged model from HuggingFace Hub (or use the local LoRA adapter).

In [ ]:
from transformers import pipeline

# Load from Hub (merged model) or local adapter
eval_model_id = hub_repo_id  # Use merged model from Hub
# eval_model_id = instruct_output_dir  # Or use local LoRA adapter

ft_pipe = pipeline(
    "image-text-to-text",
    model=eval_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

ft_pipe.model.generation_config.do_sample = False
ft_pipe.model.generation_config.pad_token_id = ft_pipe.tokenizer.eos_token_id
ft_pipe.tokenizer.padding_side = "left"

print(f"Fine-tuned model loaded from: {eval_model_id}")

### Run inference on validation samples

Test the model on examples from the VQA validation set to qualitatively assess performance.

In [ ]:
from IPython.display import display, Markdown, Image as IPImage
import io

num_eval_samples = 5  # @param {type: "number"}

for i in range(num_eval_samples):
    sample = vqa_data["validation"][i]
    messages = sample["messages"]

    # Build the prompt (system + user only, no assistant)
    prompt_messages = [m for m in messages if m["role"] != "assistant"]

    output = ft_pipe(
        text=prompt_messages,
        images=[sample["image"]],
        max_new_tokens=300,
        return_full_text=False,
    )

    predicted = output[0]["generated_text"][-1]["content"]

    # Get ground truth
    gt_msg = next(m for m in messages if m["role"] == "assistant")
    if isinstance(gt_msg["content"], list):
        gt = gt_msg["content"][0]["text"]
    else:
        gt = gt_msg["content"]

    # Get question text
    user_msg = next(m for m in messages if m["role"] == "user")
    if isinstance(user_msg["content"], list):
        question = next(c["text"] for c in user_msg["content"] if c["type"] == "text")
    else:
        question = user_msg["content"]

    display(Markdown(f"---\n\n**Sample {i+1}** | Source: `{sample['source']}` | Condition: `{sample['condition']}`"))
    display(Markdown(f"**Q:** {question}"))
    display(Markdown(f"**Ground Truth:** {gt[:300]}..."))
    display(Markdown(f"**DentalGemma:** {predicted[:300]}..."))

### Test text-only clinical reasoning

In [ ]:
from IPython.display import display, Markdown

system_instruction = (
    "You are an expert dental clinician and radiologist AI assistant. "
    "Analyze dental images and clinical information to provide accurate, "
    "evidence-based assessments. Always recommend clinical correlation and "
    "professional evaluation for definitive diagnosis."
)

prompt = (
    "A 35-year-old male presents with severe throbbing pain in the lower right "
    "molar region for 3 days. Clinical exam shows deep carious lesion on tooth #46 "
    "with tenderness to percussion. Periapical radiograph shows periapical "
    "radiolucency. Patient has no significant medical history. "
    "What is your assessment and management plan?"
)

test_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}],
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    },
]

output = ft_pipe(test_messages, max_new_tokens=512, do_sample=False)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(Markdown(f"---\n\n**[ DentalGemma ]**\n\n{response}\n\n---"))

## Next Steps

- **Build an app**: Create a Gradio/Streamlit demo for interactive dental X-ray analysis
- **Quantitative evaluation**: Run systematic metrics (BLEU, ROUGE, accuracy) on the full validation set
- **Edge deployment**: Quantize the merged model for mobile/edge inference (Edge AI Prize track)

---

**DentalGemma** — Built for the [MedGemma Impact Challenge](https://kaggle.com/competitions/med-gemma-impact-challenge) 🦷